<a href="https://colab.research.google.com/github/Jyotibrat/BotMed-Clamifision/blob/main/notebooks/Clamifision_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BotMed: Medical vs Non-Medical Text Classifier
Fine-tunes **ModernBERT-base** on your curated dataset (`train.csv` / `val.csv` / `test.csv`).

**Before running:** in Colab, go to `Runtime > Change runtime type > T4 GPU` (free tier) and save.
Run cells top to bottom, in order.

This notebook stops after training + full validation/test evaluation. Saving/downloading the
model is a separate step -- ask for that cell once you're happy with the results here.

## 1. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU, then re-run this cell.")


CUDA available: True
GPU: Tesla T4


## 2. Install dependencies
Colab already has `torch`; this installs the rest.

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


## 3. Upload your dataset
Upload `train.csv`, `val.csv`, and `test.csv` from `data/final/` (the output of your dataset builder).
A file picker will appear -- select all three at once.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Update this to the folder in your Drive that contains train.csv, val.csv, test.csv
DATA_DIR = "/content/drive/MyDrive/BotMed Binary Classifier Data"

import shutil
for fname in ["train.csv", "val.csv", "test.csv"]:
    shutil.copy(f"{DATA_DIR}/{fname}", fname)

print("Copied from Drive:", ["train.csv", "val.csv", "test.csv"])

Mounted at /content/drive
Copied from Drive: ['train.csv', 'val.csv', 'test.csv']


## 4. Load and inspect the data

In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

print("train:", train_df.shape, " val:", val_df.shape, " test:", test_df.shape)
print()
print("Train class balance:")
print(train_df["label"].value_counts())
print()
train_df.head(3)


train: (34062, 4)  val: (4258, 4)  test: (4258, 4)

Train class balance:
label
1    17031
0    17031
Name: count, dtype: int64



,text,label,source,subtopic
0,85. Br J Surg. 2020 Nov;107(12):e569-e570. doi...,1,pubmed,abstract
1,"Toby Marks rules While ""Last Train to Lhasa"" r...",0,amazon_yelp,amazon_review
2,just what I needed I need t's like this to get...,0,amazon_yelp,amazon_review


## 5. Tokenize
Loads the ModernBERT tokenizer and converts each split into a tokenized HuggingFace `Dataset`.
`max_length=256` covers almost all real user queries; PubMed/Wikipedia rows longer than that get truncated, which is fine for a topic classifier.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "answerdotai/ModernBERT-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

print(train_ds)


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/34062 [00:00<?, ? examples/s]

Map:   0%|          | 0/4258 [00:00<?, ? examples/s]

Map:   0%|          | 0/4258 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 34062
})


## 6. Load the model
`attn_implementation="sdpa"` is used instead of Flash Attention 2 -- Colab's free T4 GPU (Turing
architecture) doesn't support Flash Attention 2, which needs Ampere or newer. SDPA is fast and
works everywhere, so this avoids a crash on the free tier. If you're on an A100/L4 runtime, you
can change this to `"flash_attention_2"` for a further speed boost (after `pip install flash-attn`).

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "non_medical", 1: "medical"}
label2id = {"non_medical": 0, "medical": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    attn_implementation="sdpa",
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 7. Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


## 8. Training arguments
Batch size 16 / max_length 256 comfortably fits a free T4's 16GB VRAM for ModernBERT-base.
`fp16=True` uses mixed precision, which T4 supports well (unlike bf16, which needs Ampere+).
`eval_strategy="epoch"` means you'll see validation metrics printed after every epoch during training.

In [ ]:
from transformers import TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./botmed-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)


## 9. Train
This will print a live progress table with training loss and, after every epoch, validation loss/accuracy/precision/recall/F1.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000129,0.034268,0.996712,0.993467,1.000000,0.996723
2,0.012321,0.006008,0.998591,0.998123,0.999061,0.998592
3,0.000004,0.007929,0.998826,0.998124,0.999530,0.998827


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6387, training_loss=0.006078971315274486, metrics={'train_runtime': 2215.5106, 'train_samples_per_second': 46.123, 'train_steps_per_second': 2.883, 'total_flos': 1.7334583235682384e+16, 'train_loss': 0.006078971315274486, 'epoch': 3.0})

## 10. Full validation set evaluation
A detailed report on the **validation** set: overall metrics, a per-class precision/recall/F1
breakdown, and a confusion matrix -- all displayed directly in the cell output, not just a single
summary number.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def full_evaluation_report(trainer, dataset, df, split_name):
    print(f"{'=' * 60}")
    print(f"  {split_name.upper()} SET EVALUATION")
    print(f"{'=' * 60}\n")

    metrics = trainer.evaluate(dataset)
    print("Overall metrics:")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:25s}: {v:.4f}")
    print()

    preds_output = trainer.predict(dataset)
    preds = np.argmax(preds_output.predictions, axis=-1)
    labels = preds_output.label_ids

    print("Per-class report:")
    report = classification_report(
        labels, preds, target_names=["non_medical", "medical"], digits=4
    )
    print(report)

    cm = confusion_matrix(labels, preds)
    cm_df = pd.DataFrame(
        cm,
        index=["actual_non_medical", "actual_medical"],
        columns=["pred_non_medical", "pred_medical"],
    )
    print("Confusion matrix:")
    display(cm_df)

    return preds, labels

val_preds, val_labels = full_evaluation_report(trainer, val_ds, val_df, "validation")


  VALIDATION SET EVALUATION



Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000004,0.007929,3,0.998826,0.998124,0.999530,0.998827


Overall metrics:
  eval_loss                : 0.0079
  eval_accuracy            : 0.9988
  eval_precision           : 0.9981
  eval_recall              : 0.9995
  eval_f1                  : 0.9988



Per-class report:
              precision    recall  f1-score   support

 non_medical     0.9995    0.9981    0.9988      2129
     medical     0.9981    0.9995    0.9988      2129

    accuracy                         0.9988      4258
   macro avg     0.9988    0.9988    0.9988      4258
weighted avg     0.9988    0.9988    0.9988      4258

Confusion matrix:


,pred_non_medical,pred_medical
actual_non_medical,2125,4
actual_medical,1,2128


## 11. Full test set evaluation
Same detailed report, run on the held-out **test** set instead. Remember: this test set is drawn
from the same sources as training data (PubMed, MedQuAD, etc.), so strong numbers here don't fully
guarantee good performance on short, casual, real-user queries -- see the next cell for that check.

In [ ]:
test_preds, test_labels = full_evaluation_report(trainer, test_ds, test_df, "test")


  TEST SET EVALUATION



Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000004,0.008928,3,0.999061,0.999530,0.998591,0.999060


Overall metrics:
  eval_loss                : 0.0089
  eval_accuracy            : 0.9991
  eval_precision           : 0.9995
  eval_recall              : 0.9986
  eval_f1                  : 0.9991



Per-class report:
              precision    recall  f1-score   support

 non_medical     0.9986    0.9995    0.9991      2129
     medical     0.9995    0.9986    0.9991      2129

    accuracy                         0.9991      4258
   macro avg     0.9991    0.9991    0.9991      4258
weighted avg     0.9991    0.9991    0.9991      4258

Confusion matrix:


,pred_non_medical,pred_medical
actual_non_medical,2128,1
actual_medical,3,2126


## 12. Sanity-check on realistic casual queries
This is the check that actually matters for BotMed: short, informal, typo-prone text closer to
what real users will type, which the training data under-represents (mostly formal/clinical
sources). Add your own examples here -- this is the fastest signal for whether the register gap
we flagged earlier is actually hurting you in practice.

In [ ]:
test_queries = [
    "I've had a headache for 3 days and my vision is blurry",
    "whats the best pizza topping combo",
    "my chest hurts when i breathe in deeply, should i worry",
    "how do i reset my wifi router",
    "took 2 tylenol but fever wont go down, is that normal",
    "recommend me a good sci fi movie for tonight",
    "kid has a rash all over their arms since yesterday",
    "how to parallel park a car",
]

inputs = tokenizer(test_queries, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    preds = torch.argmax(probs, dim=-1)

results_df = pd.DataFrame({
    "query": test_queries,
    "predicted_label": [id2label[p.item()] for p in preds],
    "confidence": [f"{probs[i][preds[i]].item():.3f}" for i in range(len(test_queries))],
})
display(results_df)


,query,predicted_label,confidence
0,I've had a headache for 3 days and my vision i...,medical,1.000
1,whats the best pizza topping combo,non_medical,1.000
2,"my chest hurts when i breathe in deeply, shoul...",medical,1.000
3,how do i reset my wifi router,non_medical,0.522
4,"took 2 tylenol but fever wont go down, is that...",medical,1.000
5,recommend me a good sci fi movie for tonight,non_medical,1.000
6,kid has a rash all over their arms since yeste...,medical,1.000
7,how to parallel park a car,non_medical,1.000
